<a href="https://colab.research.google.com/github/crfrank77/DERConnect-Lab-Work/blob/main/Project_in_a_box_optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **DERConnect Project in a Box**
# **P4: Optimization**

# Repository Cloning


1. Download the zip file at this [link](https://drive.google.com/file/d/1QQpTTPLdouSDcYUP3wZ9BGnPxaXJpFU9/view?usp=drive_link)
2. Run this section by clicking the play button in the top left corner, a button that says choose files will apear below this section.
3. Select the zip file you just downloaded, it should be titled "derconnect-project-in-a-box-P4.zip"

In [ ]:
# zip file download link:
# https://drive.google.com/file/d/1QQpTTPLdouSDcYUP3wZ9BGnPxaXJpFU9/view?usp=drive_link
from google.colab import files
uploaded = files.upload()

Saving derconnect-project-in-a-box-P4.zip to derconnect-project-in-a-box-P4.zip


# Extract zip

Extract the zip file by running this section of code

In [ ]:
import zipfile

with zipfile.ZipFile("derconnect-project-in-a-box-P4.zip", "r") as zip_ref:
    zip_ref.extractall("derconnect")

%cd derconnect

/content/derconnect


# Install requirements.txt & Relevant files

This optimizer uses a few specific libraries and files to work, run these two sections to ensure they are downloaded to your colab session

In [ ]:
!pip install -r /content/derconnect/derconnect-project-in-a-box-2-optimization/requirements.txt

In [ ]:
from dotenv import load_dotenv
load_dotenv(dotenv_path="/content/derconnect/derconnect-project-in-a-box-2-optimization/Software_P4/mvp1_temp/.env")
import cvxpy as cp
import numpy as np
import pandas as pd
import os
import plotly.graph_objects as go
import time

# Load Utility functions

The utility functions do the bulk of the work in this optimizer, run this section to ensure they are also loaded to the colab session

In [ ]:
%cd /content/derconnect/derconnect-project-in-a-box-2-optimization/Software_P4/mvp1_temp
from utilities.optimize_microgrid import optimize_microgrid
from utilities.sim_results import sim_results
from utilities.plotting import plotting
from utilities.welcome import welcome

/content/derconnect/derconnect-project-in-a-box-2-optimization/Software_P4/mvp1_temp


# Optimizer


This is the optimizer! It pulls the utility functions and libraries we loaded earlier and runs them based on the situational parameters you outline.
Run this section and enter values for the rate model, case study, and bess size to get unique results on the best way to run the microgrid.
Each trial should take around a minute to run and will provide you with an output table and several interactive graphs.

In [ ]:
counter = 0
while True:
  # Welcome and input
  welcome_choices = welcome(counter=counter)
  rate_model = welcome_choices['rate_model']
  case_id = welcome_choices['case_id']
  bess_size = welcome_choices['bess_size'].strip().upper()
  counter = welcome_choices['counter']

  solar_start_day = int(os.getenv(f"CASE{case_id}_SOLAR_START_DAY"))
  demand_start_day = int(os.getenv(f"CASE{case_id}_DEMAND_START_DAY"))
  num_days = int(os.getenv(f"CASE{case_id}_NUM_DAYS"))
  output_file = str(os.getenv(f"CASE{case_id}_OUTPUT_FILE"))

  bess_capacity = float(os.getenv(f"BESS_{bess_size}_CAPACITY"))
  charge_rate = float(os.getenv(f"BESS_{bess_size}_CHARGE_RATE"))
  discharge_rate = float(os.getenv(f"BESS_{bess_size}_DISCHARGE_RATE"))
  efficiency = float(os.getenv(f"BESS_{bess_size}_EFFICIENCY"))

  %cd /content/derconnect/derconnect-project-in-a-box-2-optimization
  # Run simulation
  full_results = optimize_microgrid(solar_start_day= solar_start_day,
                                  demand_start_day= demand_start_day,
                                   num_days= num_days, rate_model = rate_model, bess_capacity = bess_capacity, charge_rate = charge_rate,
                                   discharge_rate = discharge_rate, efficiency = efficiency, enable_battery=True)
  nobess_results = optimize_microgrid(solar_start_day=solar_start_day,
                                    demand_start_day= demand_start_day,
                                     num_days= num_days,rate_model = rate_model, bess_capacity = bess_capacity, charge_rate = charge_rate,
                                     discharge_rate = discharge_rate, efficiency = efficiency, enable_battery=False)

  # Output table with hourly results and basic grid and cost info
  results_sim = sim_results(full_results=full_results,
                          nobess_results=nobess_results,
                          case_id = case_id, bess_size = bess_size, num_days=num_days,
                        output_file=output_file)

  # Plotting results
  goplot = plotting(full_results=full_results, nobess_results=nobess_results, num_days = num_days)

  repeat = input("\nWould you like to run another simulation? (yes/no): ").strip().lower()
  if repeat != "yes":
      print("\nThanks for exploring the Optimization Simulator. Goodbye!\n")
      break


  ______   ________  _______      ______                                         _    
 |_   _ `.|_   __  ||_   __ \   .' ___  |                                       / |_  
   | | `. \ | |_ \_|  | |__) | / .'   \_|  .--.   _ .--.   _ .--.  .---.  .---.`| |-' 
   | |  | | |  _| _   |  __ /  | |       / .'`\ \[ `.-. | [ `.-. |/ /__\ / /'`\]| |   
  _| |_.' /_| |__/ | _| |  \ \_\ `.___.'\| \__. | | | | |  | | | || \__.,| \__. | |,  
 |______.'|________||____| |___|`.____ .' '.__.' [___||__][___||__]'.__.''.___.'\__/  

Welcome to the Optimized Microgrid Simulator!
This simulator allows you to run simulations on a microgrid that consists of
a single building demanding power, a solar energy source, and a Battery Energy Storage System (BESS).
Through this simulator, you can select a pricing model, a speciifc case study, and a BESS size to optimize over a 30 day period.
The case studies each have different solar and demand data the program optimizes for, 
this affects how the BESS is used a


Would you like to run another simulation? (yes/no): no

Thanks for exploring the Optimization Simulator. Goodbye!



In [ ]:
 full_results


{'demand': array([268.43607843, 264.16959244, 266.81873125, 266.12496626,
        286.46157783, 361.17675507, 374.25791633, 362.17471313,
        453.78087616, 467.11243439, 496.40055084, 519.20905304,
        517.90659332, 486.92866516, 458.87880707, 457.0378418 ,
        446.74774933, 405.21967292, 392.70566264, 415.76312906,
        326.01507777, 301.64563623, 292.05463722, 292.77049875,
        283.43361101, 276.9965536 , 272.80829042, 274.15388554,
        297.01785296, 367.8019073 , 374.48016611, 369.57302174,
        451.1547699 , 476.93196869, 521.5480957 , 524.03182983,
        534.23135376, 519.06155396, 507.7605896 , 473.79199219,
        445.95666504, 411.05156511, 419.77243268, 423.99938095,
        321.29762018, 273.93370074, 273.13745692, 288.80382481,
        275.65849215, 268.75361326, 268.4011316 , 270.22398493,
        302.64196968, 349.67056474, 371.32577133, 326.00694275,
        324.17840576, 331.92749023, 375.31235504, 362.41749573,
        359.27170563, 349.7420